In [ ]:
from ocpa.objects.log.importer.ocel import factory as ocel_import_factory
from ocpa.visualization.log.variants import factory as variants_visualization_factory

import Super_Variant_Definition as SVD
import Super_Variant_Hierarchy as SVH
import Super_Variant_Visualization as SVV
import Intra_Variant_Generation as IAVG
import Intra_Variant_Summarization as IAVS
import Summarization_Selection as IASS
import Inter_Variant_Generation as IEVG
import Inter_Variant_Summarization as IEVS
import Input_Extraction_Definition as IED
import Inter_Lane_Summarization as ILS
import Inter_Lane_Alignment as ILA

from Evaluation.templates import (build_triad_hierarchy, print_hierarchy_costs)
from config import EVALUATION_LOG, PUBLICATION_LOG, PRESENTATION_LOG, TEST_LOG

VISUALIZE_INTERMEDIATE_RESULTS = False

In [ ]:
# EVALUATION Settings
ILA.ALIGN = False
filename = PUBLICATION_LOG # "EventLogs/BPI2017-Top10.jsonocel"  # EVALUATION_LOG

# Import Log
ocel = ocel_import_factory.apply(file_path=filename)

## Intra-Variant Summarization

In [ ]:
### FUNCTION IED.get_unique_summarizations_from_process()
from ocpa.visualization.log.variants import factory as variants_visualization_factory

# -> Division into discrete steps already happens here, viz package part of process discovery in ocpa
# {Activity: [start index, end index], [other appearances / intersections]}
variant_layout = variants_visualization_factory.apply(ocel)
print("Base Variants from ocel:\n", variant_layout, "\n\n\n")
variants = []

# Actually iterated over all variants
sample_variant = 0

# Extract lanes from variant
extracted_variant = IED.extract_lanes(variant_layout[ocel.variants[sample_variant]], ocel.variant_frequencies[sample_variant])
variants.append(extracted_variant)
print("Extracted Variant:\n", extracted_variant, "\n\n\n")

# summarize the variant, then added to all_summarizations
extracted_summarizations = IAVS.within_variant_summarization(extracted_variant, print_results=False)
sample_summarization = extracted_summarizations[0]
# sample_summarization added to list of all summarizations
print("Extracted Summarization:\n", sample_summarization, "\n\n\n")

encoding = sample_summarization.encode_lexicographically()
# encoding used as key in dict to store all unique summarizations
# as values, add for index 0 the variant number and for index 1 the summarization 
# -> {encoding: {0: [variant_number], 1: [summarization]}}
# encoding used als key in set to store all unique summarizations 
# -> {(encoding, [summarization])}
print("Encoding:\n", encoding, "\n\n\n")

### FUNCTION IAVG.complete_intra_variant_summarization_from_process
# BAD SCALABILITY this line
universe, summ_dict = IAVG.get_unique_summarizations_from_process(ocel, print_results=True)
# Sorts the unique summarizations into subsets based on the variants that they summarize.
all_summ, summ_per_variant = IAVG.__determine_subsets(universe, summ_dict)
print("All Summarizations:\n", all_summ[0], "\n")
print("Per Variant Dict:\n", summ_per_variant[0], "\n")
print("Per Encoding Dict:\n", summ_dict, "\n\n\n")





# Generate and select intra-variant summarizations
all_summarizations, per_variant_dict, per_encoding_dict = IAVG.complete_intra_variant_summarization_from_process(ocel, print_results=False)

intra_variant_summarizations = IASS.intra_variant_summarization_selection(all_summarizations, per_variant_dict, per_encoding_dict)
print("Intra Variant Summarizations:\n", intra_variant_summarizations[0], "\n\n\n")

try:
    # EVALUATION Settings
    # Cleansing with actual frequencies
    global_frequencies = [0.069, 0.05, 0.047, 0.03, 0.025, 0.019, 0.019, 0.018, 0.017, 0.016]
    for i in range(len(intra_variant_summarizations)):
        intra_variant_summarizations[i].frequency = global_frequencies[i]
    
    # Visualize selected intra-variant summarizations
    if VISUALIZE_INTERMEDIATE_RESULTS:
        for i in range(len(intra_variant_summarizations)):
            SVV.visualize_super_variant(intra_variant_summarizations[i])
    
    if VISUALIZE_INTERMEDIATE_RESULTS:
        # Visualize intra-variant summarizations of variant v_3
        SVV.MARK_INCORRECT_INTERACTIONS = False
        SVV.visualize_variant(all_summarizations[3][1][1][0],3)
        SVV.visualize_variant(all_summarizations[4][1][1][0],3)
        SVV.MARK_INCORRECT_INTERACTIONS = True
except:
    for x in intra_variant_summarizations:
        print(x)

In [ ]:
first = all_summarizations[3]
SVV.visualize_super_variant(intra_variant_summarizations[0])

superVarNr = first[0]
superVarStr = first[1][0]
superVarRef = first[1][1]

print("Super Variant Number: ", superVarNr)
print("Super Variant String:\n", superVarStr)

@SuperLane.encode_lexicographically()

a 0 -> Lane a, instance 0

CAR 1 -> Lane instance is executed exactly once

Constructs: -> [startIndex:EndIndex]
- IP = Interaction Point
- CO = CommonConstruct / "Plain Activity"
- CH = ChoiceConstruct
- OP = OptionalConstruct

# Inter-Variant Summarization

In [ ]:
# MODE 1 - Frequency Distribution and No Nested Structures
IEVG.NESTED_STRUCTURES = False
initial_super_variants = intra_variant_summarizations

# Generating 3 hierarchies from the initial super variants with uniform frequency distribution
hierarchies, final_super_variants = IEVG.generate_super_variant_hierarchy(
    initial_super_variants, 3, frequency_distribution_type=IEVG.Distribution.UNIFORM)

print("Hierarchies", hierarchies, "\n\n\n")
print("Final super variants", final_super_variants, "\n\n\n")

# Visualizing each hierarchy top-down
for final_super_variant in final_super_variants[0]:
    SVH.explore_hierarchy_top_down(final_super_variant)

print_hierarchy_costs(hierarchies)

In [ ]:
print(hierarchies[0], str([0]))
SVH.explore_hierarchy_top_down(final_super_variants[0][0])

In [ ]:
print(hierarchies[0][0])